In [1]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from google.colab import drive, runtime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [3]:
# Core training config.
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    "max_seq_length": 4096, #Provides enough room for input tokens and output tokens
    "lora_r": 64, #Different lora values did not have a noticeable impact on training results.
    "lora_alpha": 128,
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 32,
    "gradient_accumulation_steps": 2,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 600,
    "max_train_samples_per_source": 50000,
    "eval_size": 0.02,
    "output_dir": "final-model",
}

SYSTEM_PROMPT = (
    "You are an SVG code generator. Given a description, output only valid SVG code, nothing else. "
    "Only use these elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, "
    "desc, style, pattern, marker, filter."
    "Keep the final SVG code strictly under 15000 characters."
    "Always use exactly these attributes in the opening tag: width=\"256\" height=\"256\" viewBox=\"0 0 200 200\""
)

CONFIG

{'model_name': 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
 'max_seq_length': 4096,
 'lora_r': 64,
 'lora_alpha': 128,
 'learning_rate': 0.0002,
 'num_train_epochs': 3,
 'per_device_train_batch_size': 32,
 'gradient_accumulation_steps': 2,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 600,
 'max_train_samples_per_source': 50000,
 'eval_size': 0.02,
 'output_dir': 'final-model'}

In [4]:
#Mound to drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/DL-midterm-2026')

#Read training set, drop id column, convert prompt and svg columns to string type
df = pd.read_csv("train.csv")
df.drop('id', axis = 1)
df = df[['prompt', 'svg']].astype('string')
df['svg'] = df['svg'].str.replace(r'\d+\.\d+', lambda m: f"{float(m.group()):.1f}".rstrip('0').rstrip('.'), regex = True)
df['svg'].str.len().describe()
df['svg'].str.extract(r'(viewBox="[^"]+")')[0].value_counts()
df = df[df['svg'].str.contains(r'viewBox="0 0 200 200"', regex = True, na = False)]
df['svg'].str.extract(r'(viewBox="[^"]+)"')[0].value_counts()
lengths = df['svg'].str.len()
threshold = lengths.quantile(0.99)
df = df[lengths <= threshold]
max_row = df.loc[[df['svg'].str.len().idxmax()]]
max_row.head(1)

Mounted at /content/drive


,prompt,svg
27010,The image features a blue megaphone with a bla...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [5]:
#Filter out samples that use disallowed tags
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan", "title",
    "desc", "style", "pattern", "marker", "filter"
}

def has_only_allowed_tags(svg_text):
    """Parses SVG and returns False if any tag is not in ALLOWED_TAGS."""
    if pd.isna(svg_text) or not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        for elem in root.iter():
            # Strip XML namespaces (e.g., {http://www.w3.org/2000/svg}path -> path)
            tag_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
            if tag_name not in ALLOWED_TAGS:
                return False
        return True
    except ET.ParseError:
        # Drop rows with invalid/unparsable XML
        return False

print(f"Rows before tag filtering: {len(df)}")
df = df[df['svg'].apply(has_only_allowed_tags)]
print(f"Rows after tag filtering: {len(df)}")
df['svg'].str.len().describe()

Rows before tag filtering: 40972
Rows after tag filtering: 40972


,svg
count,40972.0
mean,1123.974104
std,559.625465
min,112.0
25%,686.0
50%,1015.0
75%,1471.0
max,2709.0


In [6]:
from unsloth import FastLanguageModel
#The prior operations have cut the average size of the svg strings in half, decreased the max svg size from 15,937 characters to 2971, and standardized the dimensions of the svg viewboxes.
#This will substantially decrease wasteful token consumption for our model. We are now ready to convert these rows into the conversational format that the model expects.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

#Modify dataframe rows for model use
def build_conversations(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["svg"]}
    ]
    formatted_string = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
    return formatted_string

df['text'] = df.apply(build_conversations, axis = 1)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

In [7]:
#Filter out prompts that consume 2048 tokens or more
def count_tokens(text):
    return len(tokenizer(text)["input_ids"])
df['token_length'] = df['text'].apply(count_tokens)
df = df[df['token_length'] < 2046].copy()

#Convert pandas dataframe to hugging face dataset and split into training and eval sets
hf_dataset = Dataset.from_pandas(df)

split_dataset = hf_dataset.train_test_split(test_size = CONFIG['eval_size'], seed = SEED)
train_ds = split_dataset['train']
eval_ds = split_dataset['test']

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
print(train_ds[:1])

Train rows: 37781
Eval rows: 772
{'prompt': ['A simple line drawing of a box with a lid.'], 'svg': ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#151515" fill-opacity="1"  filling="0" d="M145.9 180.1 L53.8 180.1 C40.5 180.1 29.7 169.3 29.7 156 L29.7 98.9 C29.7 95.1 32.8 92.1 36.5 92.1 C40.3 92.1 43.4 95.1 43.4 98.9 L43.4 156 C43.4 161.7 48 166.4 53.8 166.4 L145.9 166.4 C151.7 166.4 156.3 161.7 156.3 156 L156.3 98.9 C156.3 95.1 159.4 92.1 163.2 92.1 C167 92.1 170 95.1 170 98.9 L170 156 C170 169.3 159.2 180.1 145.9 180.1 Z"></path>\n<path fill="#151515" fill-opacity="1"  filling="0" d="M116.8 138.8 L82.9 138.8 C79.1 138.8 76.1 135.7 76.1 131.9 C76.1 128.1 79.1 125.1 82.9 125.1 L116.8 125.1 C120.6 125.1 123.7 128.1 123.7 131.9 C123.7 135.7 120.6 138.8 116.8 138.8 Z M150.5 85.1 L49.5 85.1 C42 85.1 35.3 81 31.9 74.3 C28.5 67.6 29.1 59.7 33.5 53.6 L52.1 28.1 C55.8 23 61.8 19.9 68.1 19.9 L131.9 19.9 C138.2 19.9 144.2 23 147.9 28.1 L16

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

Unsloth 2026.3.18 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import warnings
from transformers.trainer_utils import get_last_checkpoint

# Suppress specific Hugging Face FutureWarnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="transformers.*"
)

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

latest_checkpoint = get_last_checkpoint(CONFIG['output_dir'])
if latest_checkpoint is not None:
    print(f"Resuming from checkpoint: {latest_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("No checkpoint found. Starting training from scratch.")
    train_result = trainer.train()

train_result

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/37781 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/772 [00:00<?, ? examples/s]

Resuming from checkpoint: final-model/checkpoint-1884


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 37,781 | Num Epochs = 3 | Total steps = 3,543
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 119,734,272 of 3,205,672,960 (3.74% trained)
	per_device_train_batch_size: 32 (from args) != 16 (from trainer_state.json)


Step,Training Loss,Validation Loss
1900,0.350400,0.300096


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])

tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

print("Disconnecting and terminating Colab runtime...")
runtime.unassign()
